In [1]:
import sys; sys.path.insert(0, "../src")
import itertools, numpy as np, pandas as pd
from elo import add_elo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

df0 = pd.read_csv("../data/processed/laliga_matches.csv",
                  parse_dates=["Date"], dtype={"Season": str})
df0["y"] = df0["FTR"].map({"H": 0, "D": 1, "A": 2})

TR_FIT = ["1617","1718","1819","1920","2021"]   # ajustar la logística
VAL    = ["2122","2223"]                         # elegir parámetros aquí
TEST   = ["2324","2425","2526"]                  # NO se toca

def score(k, home_adv, regress, fit=TR_FIT, val=VAL):
    d = add_elo(df0, k=k, home_adv=home_adv, regress=regress)
    a, b = d[d["Season"].isin(fit)], d[d["Season"].isin(val)]
    m = LogisticRegression(max_iter=1000).fit(a[["elo_diff"]], a["y"])
    return log_loss(b["y"], m.predict_proba(b[["elo_diff"]]), labels=[0, 1, 2])

grid = list(itertools.product([5, 10, 15, 20, 30, 40],      # k
                              [0, 30, 60, 100],             # home_adv
                              [0.0, 0.15, 0.25, 0.4, 0.6])) # regress
res = [(k, h, r, score(k, h, r)) for k, h, r in grid]
res = pd.DataFrame(res, columns=["k", "home_adv", "regress", "val_logloss"])
print(res.sort_values("val_logloss").head(10).round(4))
print("\nParámetros actuales (20, 60, 0.25):",
      round(score(20, 60, 0.25), 4))

     k  home_adv  regress  val_logloss
75  20       100     0.00       1.0056
70  20        60     0.00       1.0059
55  15       100     0.00       1.0060
50  15        60     0.00       1.0061
65  20        30     0.00       1.0062
56  15       100     0.15       1.0062
45  15        30     0.00       1.0062
40  15         0     0.00       1.0063
60  20         0     0.00       1.0064
51  15        60     0.15       1.0064

Parámetros actuales (20, 60, 0.25): 1.0088


In [2]:
d_nuevo = add_elo(df0, k=20, home_adv=60, regress=0.0)
d_viejo = add_elo(df0, k=20, home_adv=60, regress=0.25)

def test_ll(d):
    tr = d[d["Season"].isin(TR_FIT + VAL)]
    te = d[d["Season"].isin(TEST)]
    m = LogisticRegression(max_iter=1000).fit(tr[["elo_diff"]], tr["y"])
    return log_loss(te["y"], m.predict_proba(te[["elo_diff"]]), labels=[0, 1, 2])

print("Elo antiguo (regress 0.25):", round(test_ll(d_viejo), 4))
print("Elo nuevo   (regress 0.00):", round(test_ll(d_nuevo), 4))

Elo antiguo (regress 0.25): 0.9746
Elo nuevo   (regress 0.00): 0.9732


## Ajuste de los parámetros del Elo

Parámetros (K, ventaja local, regresión a la media) fijados inicialmente a
mano. Búsqueda en rejilla de 120 combinaciones con validación temporal:
la logística se ajusta con 2016/17-2020/21 y se elige en 2021/22-2022/23.
El test (2023/24-2025/26) no interviene en la selección.

**Validación:** mejor combinación (K=20, ventaja=100, regresión=0) con
log-loss 1,0056 frente a 1,0088 de los parámetros iniciales. Las diez
mejores combinaciones están dentro de 0,001 entre sí. Patrón consistente:
`regress = 0` en 8 de las 10 mejores. La ventaja local apenas influye
(la regresión logística ya la absorbe).

**Elección:** K=20, ventaja=60, regresión=0 (configuración simple dentro de
la zona estable, sin elegir la esquina de la rejilla).

**Test (una única evaluación):**

| Elo | Log-loss |
|---|---|
| Inicial (regresión 0,25) | 0,9746 |
| Ajustado (regresión 0) | 0,9732 |

**Conclusión:** la mejora (0,0014) es muy inferior al ruido del test
(IC bootstrap de ±0,011 en diferencias pareadas). El ajuste de parámetros no
aporta una mejora medible; el Elo ya está cerca de su techo con una sola
variable.